In [1]:
x = [[1,2],[3,4],[5,6],[7,8]]
y = [[3],[7],[11],[15]]

In [2]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [3]:
class MyDataset(Dataset):
    def __init__(self, x, y):
        self.x = torch.tensor(x).float().to(device)
        self.y = torch.tensor(y).float().to(device)
    
    def __getitem__(self, ix):
        return self.x[ix], self.y[ix]

    def __len__(self):
        return len(self.x)

In [4]:
ds = MyDataset(x, y)
dl = DataLoader(ds, batch_size=2, shuffle=True)

In [5]:
model = nn.Sequential(
    nn.Linear(2, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
).to(device)

In [6]:
loss_func = nn.MSELoss()

from torch.optim import SGD

opt = SGD(model.parameters(), lr = 0.001)

import time

loss_history = []
start = time.time()
for _ in range(50):
    for ix, iy in dl:
        opt.zero_grad()
        loss_value = loss_func(model(ix),iy)
        loss_value.backward()
        opt.step()
        loss_history.append(loss_value.item())
end = time.time()
print(end - start)

0.15625309944152832


In [7]:
model.state_dict()

OrderedDict([('0.weight',
              tensor([[ 0.7243,  0.3515],
                      [ 0.8270,  0.6759],
                      [-0.4068, -0.6767],
                      [ 0.6734, -0.5600],
                      [ 0.6654,  0.0320],
                      [ 0.5469, -0.3493],
                      [ 0.3829, -0.5401],
                      [ 0.6562, -0.6515]])),
             ('0.bias',
              tensor([ 0.7250, -0.1501,  0.1840,  0.1008,  0.3468,  0.4100,  0.1649,  0.5442])),
             ('2.weight',
              tensor([[ 0.6253,  0.7151,  0.0137,  0.0169,  0.3471, -0.0200,  0.0082, -0.2557]])),
             ('2.bias', tensor([-0.1071]))])

In [8]:
torch.save(model.to('cpu').state_dict(), 'mymodel.pth')

In [9]:
model = nn.Sequential(
    nn.Linear(2, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
).to(device)

In [10]:
state_dict = torch.load('mymodel.pth')

In [11]:
model.load_state_dict(state_dict)
model.to(device)

val = [[8,9],[10,11],[1.5,2.5]]

model(torch.tensor(val).float().to(device))

tensor([[16.9644],
        [20.9393],
        [ 4.0506]], grad_fn=<AddmmBackward0>)